In [ ]:
import json

import pandas as pd
from IPython.display import display
from matplotlib_inline.backend_inline import set_matplotlib_formats

from emu_renewal.constants import DATA_PATH, FULL_RUN
from emu_renewal.outputs import (
    get_median_posterior,
    get_param_vals_by_analysis,
    get_prop_better,
)
from emu_renewal.plotting import plot_policy_mobility_floor_pairs
from emu_renewal.utils import get_analysis_paths, get_cont_of_country

set_matplotlib_formats("svg")


In [ ]:
POLICY = "oxcgrt_floored"
BASELINE = "no_scaling"
MOB_ANALYSES = ["g_mob", "fb_visited_mob"]

all_countries = json.load(open(DATA_PATH / "config/oxcgrt_included.json"))
analysis_paths = get_analysis_paths(FULL_RUN, all_countries)

records = {}
for iso3, paths in analysis_paths.items():
    if POLICY not in paths or BASELINE not in paths:
        continue
    if not any(m in paths for m in MOB_ANALYSES):
        continue
    c_paths = {k: paths[k] for k in (BASELINE, POLICY)}
    disp = get_param_vals_by_analysis("dispersion_proc", c_paths)
    row = {
        "continent": get_cont_of_country(iso3),
        "floor_policy": get_median_posterior(paths[POLICY], "scale_floor"),
        "prop_better_policy": get_prop_better(disp, POLICY, BASELINE),
    }
    for mob in MOB_ANALYSES:
        if mob in paths:
            row[f"floor_{mob}"] = get_median_posterior(paths[mob], "scale_floor")
            mob_disp = get_param_vals_by_analysis(
                "dispersion_proc", {BASELINE: paths[BASELINE], mob: paths[mob]}
            )
            row[f"prop_better_{mob}"] = get_prop_better(mob_disp, mob, BASELINE)
    records[iso3] = row

summary = pd.DataFrame.from_dict(records, orient="index")
# Keep countries with evidence of lower dispersion under OxCGRT floored;
# each panel further restricts to mobility-supported countries
summary = summary.loc[summary["prop_better_policy"] >= 0.5]


In [ ]:
fig = plot_policy_mobility_floor_pairs(summary, MOB_ANALYSES)
display(fig)


In [ ]:
fig.savefig("figure_8.svg", bbox_inches="tight")
